In [6]:
import pandas as pd
import duckdb
from pathlib import Path
import openpyxl
print("openpyxl is working") # for reading excel files

import matplotlib.pyplot as plt
import matplotlib.dates as mdates


openpyxl is working


In [7]:
df = pd.read_csv("../data/processed/netflix.csv")

# Register in DuckDB
duckdb.register("netflix",df)
df.shape


(458338, 15)

In [3]:
df.head()

,Unnamed: 0,country_name,country_iso2,week,country_category,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,runtime,global_weekly_views,global_cumulative_weeks_in_top_10
0,0,Argentina,AR,2026-03-15,Films,1,War Machine,NaN,2,Films (English),1.0,80600000.0,109.002,44400000.0,2.0
1,1,Argentina,AR,2026-03-15,Films,2,Strangers in the Park,NaN,2,Films (Non-English),10.0,1600000.0,115.998,800000.0,2.0
2,2,Argentina,AR,2026-03-15,Films,3,Joker: Folie à Deux,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN
3,3,Argentina,AR,2026-03-15,Films,4,Trolls Band Together,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN
4,4,Argentina,AR,2026-03-15,Films,5,Double Jeopardy,NaN,1,Films (English),5.0,6200000.0,106.002,3500000.0,1.0


In [4]:
duckdb.sql("""
DESCRIBE netflix
""").df()


,column_name,column_type,null,key,default,extra
0,Unnamed: 0,BIGINT,YES,None,None,None
1,country_name,VARCHAR,YES,None,None,None
2,country_iso2,VARCHAR,YES,None,None,None
3,week,VARCHAR,YES,None,None,None
4,country_category,VARCHAR,YES,None,None,None
5,country_weekly_rank,BIGINT,YES,None,None,None
6,show_title,VARCHAR,YES,None,None,None
7,season_title,VARCHAR,YES,None,None,None
8,country_cumulative_weeks_in_top_10,BIGINT,YES,None,None,None
9,global_category,VARCHAR,YES,None,None,None


## Dataset Checks
Validate structure first before analysis

In [5]:
duckdb.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT show_title) AS unique_show_titles,
    COUNT(DISTINCT week) AS unique_weeks,
    COUNT(DISTINCT country_name) AS unique_countries         
FROM netflix
""").df()


,total_rows,unique_show_titles,unique_weeks,unique_countries
0,458338,11060,246,94


Countries count by activity

In [6]:
duckdb.sql("""
    SELECT
    country_name,
    COUNT(*) AS appearances
FROM netflix
GROUP BY country_name
ORDER BY appearances DESC
""").df()


,country_name,appearances
0,Argentina,4922
1,El Salvador,4922
2,Venezuela,4922
3,Costa Rica,4922
4,Mexico,4922
...,...,...
89,United States,4920
90,Hong Kong,4920
91,Ireland,4920
92,Oman,4920


Most Persistent Show Title

In [7]:
duckdb.sql("""
    SELECT
    country_name,
    COUNT(*) AS appearances
FROM netflix
GROUP BY country_name
ORDER BY appearances DESC
""").df()


,country_name,appearances
0,Dominican Republic,4922
1,Honduras,4922
2,Costa Rica,4922
3,Mexico,4922
4,Peru,4922
...,...,...
89,Lebanon,4920
90,Bahrain,4920
91,Bangladesh,4920
92,Vietnam,4920


Best Performing Titles

In [8]:
duckdb.sql("""
    SELECT
    show_title,
    AVG(country_weekly_rank) AS avg_rank
FROM netflix
GROUP BY show_title
ORDER BY avg_rank ASC
LIMIT 20
""").df()


,show_title,avg_rank
0,Boy,1.000000
1,Andragogy,1.000000
2,The Call,1.000000
3,Oi Vita Mia,1.000000
4,How To Choose the Right Husband?,1.000000
5,Our Secret Diary,1.000000
6,Daddyku Gangster,1.000000
7,War Machine,1.107527
8,Dear Nathan Thank You Salma,1.500000
9,Banduan,1.500000


### UX Team's Choice of Film & TV 
- UX has decided to only use 1 Swedish Film and 1 Swedish TV Series to use for their Dashboard
- Swedish TV Series = Young Royals
- Swedish Films = The Jönsson Gang Returns

In [9]:
duckdb.sql("""
    SELECT *
FROM netflix
WHERE show_title = 'Young Royals'
""").df()

,Unnamed: 0,country_name,country_iso2,week,country_category,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,runtime,global_weekly_views,global_cumulative_weeks_in_top_10
0,2100,Argentina,AR,2024-03-17,TV,9,Young Royals,Young Royals: Season 3,1,TV (Non-English),5.0,11500000.0,237.0,2900000.0,1.0
1,11922,Austria,AT,2024-03-24,TV,10,Young Royals,Young Royals: Season 3,2,TV (Non-English),9.0,7800000.0,294.0,1600000.0,2.0
2,11937,Austria,AT,2024-03-17,TV,5,Young Royals,Young Royals: Season 3,1,TV (Non-English),5.0,11500000.0,237.0,2900000.0,1.0
3,13360,Austria,AT,2022-11-06,TV,8,Young Royals,Young Royals: Season 2,1,TV (Non-English),3.0,18860000.0,NaN,NaN,1.0
4,31605,Belgium,BE,2024-03-24,TV,10,Young Royals,Young Royals: Season 3,2,TV (Non-English),9.0,7800000.0,294.0,1600000.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,425992,Ukraine,UA,2024-03-17,TV,9,Young Royals,Young Royals: Season 1,3,None,NaN,NaN,NaN,NaN,NaN
123,427409,Ukraine,UA,2022-11-06,TV,6,Young Royals,Young Royals: Season 2,1,TV (Non-English),3.0,18860000.0,NaN,NaN,1.0
124,428772,Ukraine,UA,2021-07-18,TV,9,Young Royals,Young Royals: Season 1,2,None,NaN,NaN,NaN,NaN,NaN
125,428787,Ukraine,UA,2021-07-11,TV,4,Young Royals,Young Royals: Season 1,1,TV (Non-English),8.0,9820000.0,NaN,NaN,1.0


In [8]:
duckdb.sql("""
    SELECT *
FROM netflix
WHERE show_title = 'The Jönsson Gang Returns'
""").df()

,Unnamed: 0,country_name,country_iso2,week,country_category,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,runtime,global_weekly_views,global_cumulative_weeks_in_top_10
0,395315,Sweden,SE,2025-04-20,Films,7,The Jönsson Gang Returns,None,6,None,NaN,NaN,NaN,NaN,NaN
1,395355,Sweden,SE,2025-04-06,Films,7,The Jönsson Gang Returns,None,5,None,NaN,NaN,NaN,NaN,NaN
2,395372,Sweden,SE,2025-03-30,Films,4,The Jönsson Gang Returns,None,4,None,NaN,NaN,NaN,NaN,NaN
3,395391,Sweden,SE,2025-03-23,Films,3,The Jönsson Gang Returns,None,3,None,NaN,NaN,NaN,NaN,NaN
4,395409,Sweden,SE,2025-03-16,Films,1,The Jönsson Gang Returns,None,2,None,NaN,NaN,NaN,NaN,NaN
5,395429,Sweden,SE,2025-03-09,Films,1,The Jönsson Gang Returns,None,1,None,NaN,NaN,NaN,NaN,NaN


# Netflix EDA | show title centric 

## Top Performers
What are the Top 10 titles per category per year?

In [ ]:
duckdb.sql("""
WITH base AS (
    SELECT 
        DATE_TRUNC('year', STRPTIME(week, '%Y-%m-%d')) AS year,
        global_category,
        show_title,
        COUNT(*) AS appearances,
        AVG(country_weekly_rank) AS avg_rank,
        MAX(country_cumulative_weeks_in_top_10) AS longevity
    FROM netflix
    GROUP BY 
        year,
        global_category,
        show_title
),

ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY year, global_category
            ORDER BY 
                appearances DESC,        -- primary KPI
                avg_rank ASC             -- tie-breaker (lower is better)
        ) AS rank_in_category_year
    FROM base
)

SELECT *
FROM ranked
WHERE rank_in_category_year
ORDER BY year, global_category, rank_in_category_year
""").df()

,year,global_category,show_title,appearances,avg_rank,longevity,rank_in_category_year
0,2021-01-01,Films (English),Red Notice,581,3.399312,7,1
1,2021-01-01,Films (English),Army of Thieves,336,2.964286,4,2
2,2021-01-01,Films (English),Love Hard,296,4.195946,5,3
3,2021-01-01,Films (English),Kate,287,3.233449,4,4
4,2021-01-01,Films (English),The Guilty,285,2.343860,4,5
...,...,...,...,...,...,...,...
19357,2026-01-01,None,You Don't Mess with the Zohan,1,10.000000,1,987
19358,2026-01-01,None,Members Only: Palm Beach,1,10.000000,1,988
19359,2026-01-01,None,Distorted,1,10.000000,1,989
19360,2026-01-01,None,Risen,1,10.000000,1,990
